# 01. 定义并构建基因组 contexts

本 notebook 将构建用于富集分析的基因组 contexts


In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

data_dir = os.path.join(os.getcwd(), "data")
source_dir = os.path.join(data_dir, "source")
results_dir = os.path.join(data_dir, "results")
run_dir = os.path.join(os.getcwd(), "run")

## 1. 定义site-level contexts

### 1.1 VCF文件质控

In [2]:
landrace_vcf_file = os.path.join(data_dir, "source", "landraces.vcf.gz")

提取一个MAF大于0.05的VCF文件，作为后续大部分分析的基础。

In [3]:
from src.utils import have_file

min_maf = 0.05
max_het = 0.05
max_miss = 0.5

qc_output_vcf_file = os.path.join(data_dir, "results", f"landraces_qc_miss{max_miss}_maf{min_maf}_het{max_het}.vcf.gz")
qc_output_vcf_stat_h5 = os.path.join(data_dir, "results", f"landraces_qc_miss{max_miss}_maf{min_maf}_het{max_het}.vcf.gz.var_stat.h5")
if not have_file(qc_output_vcf_file):
    from src.vcf import quality_control_vcf
    quality_control_vcf(landrace_vcf_file, qc_output_vcf_file, min_maf=min_maf, max_het=max_het, max_miss=max_miss)
if not have_file(qc_output_vcf_stat_h5):
    from src.vcf import build_var_stat_table
    build_var_stat_table(qc_output_vcf_file, qc_output_vcf_stat_h5)

再构建一个MAF大于0.01，但仅包括CDS区域的VCF文件，作为后续LoF分析的基础。

In [4]:
min_maf = 0.01
max_het = 0.05
max_miss = 0.5

ref_gff_file = os.path.join(data_dir, "source", "Sbicolor_730_v5.1.gene_exons.gff3.gz")
cds_output_vcf_file = os.path.join(data_dir, "results", f"landraces_cds_miss{max_miss}_maf{min_maf}_het{max_het}.vcf.gz")
cds_output_vcf_file_stat_h5 = os.path.join(data_dir, "results", f"landraces_cds_miss{max_miss}_maf{min_maf}_het{max_het}.vcf.gz.var_stat.h5")
if not have_file(cds_output_vcf_file):
    from src.vcf import extract_cds_vcf
    extract_cds_vcf(landrace_vcf_file, ref_gff_file, cds_output_vcf_file)
if not have_file(cds_output_vcf_file_stat_h5):
    from src.vcf import build_var_stat_table
    build_var_stat_table(cds_output_vcf_file, cds_output_vcf_file_stat_h5)

### 1.2 对全基因组中 MAF大于0.05的位点进行context注释

获取位点列表

In [5]:
from src.contexts import load_site_info_df
import pandas as pd

site_contexts_df_h5 = os.path.join(data_dir, "results", f"site_contexts.genome.maf0.05.h5")

if not have_file(site_contexts_df_h5):
    site_contexts_df = load_site_info_df(qc_output_vcf_stat_h5)
    site_contexts_df = site_contexts_df[['ID', 'CHROM', 'POS', 'MAF']]
    site_contexts_df.to_hdf(site_contexts_df_h5, key='df')
else:
    site_contexts_df = pd.read_hdf(site_contexts_df_h5, key='df')

contexts_list = [i for i in site_contexts_df.columns.tolist() if i not in ['ID', 'CHROM', 'POS', 'MAF']]
print(f" {len(contexts_list)} contexts have already been recorded in site_contexts_df")

site_contexts_df

 30 contexts have already been recorded in site_contexts_df


,ID,CHROM,POS,MAF,gene_body,genic_2kb,genic_proximal,genic_15kb,TE,other_intergenic,...,high_pi,maf_0.05_0.1,maf_0.1_0.2,maf_0.2_0.3,maf_0.3_0.4,maf_0.4_0.5,conserved_sites,neutral_sites,accelerated_sites,non_CEs
0,SNP_Chr01_20650,Chr01,20650,0.230248,False,True,True,True,False,False,...,True,False,False,True,False,False,False,False,False,True
1,SNP_Chr01_20658,Chr01,20658,0.229120,False,True,True,True,False,False,...,True,False,False,True,False,False,False,False,False,True
2,INDEL_Chr01_20702,Chr01,20702,0.230248,False,True,True,True,False,False,...,True,False,False,True,False,False,False,False,False,True
3,SNP_Chr01_20730,Chr01,20730,0.284424,False,True,True,True,False,False,...,True,False,False,True,False,False,False,False,False,True
4,INDEL_Chr01_20736,Chr01,20736,0.178330,False,True,True,True,False,False,...,True,False,True,False,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6452573,SNP_Chr10_62843420,Chr10,62843420,0.059819,True,True,True,True,False,False,...,False,True,False,False,False,False,True,False,False,False
6452574,SNP_Chr10_62843678,Chr10,62843678,0.075621,False,True,True,True,False,False,...,False,True,False,False,False,False,False,True,False,False
6452575,SNP_Chr10_62843803,Chr10,62843803,0.059819,False,True,True,True,False,False,...,False,True,False,False,False,False,True,False,False,False
6452576,SNP_Chr10_62843840,Chr10,62843840,0.058691,False,True,True,True,False,False,...,False,True,False,False,False,False,False,True,False,False


#### Genomic Composition & Gene Structure

In [6]:
# genomic composition contexts

genomic_composition_contexts = ['gene_body', 'genic_2kb', 'genic_proximal', "genic_15kb", "TE", "other_intergenic"]
ref_gff_file = os.path.join(data_dir, "source", "Sbicolor_730_v5.1.gene_exons.gff3.gz")
ref_repeat_gff_file = os.path.join(data_dir, "source", "Sbicolor_730_v5.1.repeatmasked_assembly_v5.0.gff3.gz")

if not all([c in contexts_list for c in genomic_composition_contexts]):
    from src.contexts import get_genomic_composition_contexts
    site_contexts_df = get_genomic_composition_contexts(site_contexts_df, ref_gff_file, ref_repeat_gff_file)
    site_contexts_df.to_hdf(site_contexts_df_h5, key='df')

# gene structure contexts

gene_structure_contexts = ['upstream_2kb', "five_prime_UTR", 'CDS', 'intron', "three_prime_UTR", 'downstream_2kb']
ref_gff_file = os.path.join(data_dir, "source", "Sbicolor_730_v5.1.gene_exons.gff3.gz")

if not all([c in contexts_list for c in gene_structure_contexts]):
    from src.contexts import get_gene_struc_contexts
    site_contexts_df = get_gene_struc_contexts(site_contexts_df, ref_gff_file, gene_flank_size=2000)
    site_contexts_df.to_hdf(site_contexts_df_h5, key='df')

#### Coding Consequences

In [7]:
coding_consequence_contexts = ['missense', 'synonymous', 'putative_LoF']
cds_snpeff_vcf_file = cds_output_vcf_file
if not all([c in contexts_list for c in coding_consequence_contexts]):
    from src.contexts import get_coding_mutation_contexts
    site_contexts_df = get_coding_mutation_contexts(site_contexts_df, cds_snpeff_vcf_file)
    site_contexts_df.to_hdf(site_contexts_df_h5, key='df')

#### ACRs

ACR data were obtained from Supplementary Table 12 (“Identified ACRs in Sorghum bicolor”) of Lu et al. (2019). DOI: 10.1038/s41477-019-0548-z.

In [8]:
acrs_contexts = ['ACR']
acrs_v5_bed_file = os.path.join(data_dir, "source", "ACRs.v5.bed.gz")
if not all([c in contexts_list for c in acrs_contexts]):
    from src.contexts import get_bed_contexts
    site_contexts_df = get_bed_contexts(site_contexts_df, acrs_v5_bed_file, context_name='ACR')
    site_contexts_df.to_hdf(site_contexts_df_h5, key='df')

#### UMRs

UMR data were obtained from Dataset_S08 of Lu et al. (2019). DOI: 10.1073/pnas.2010250117.

In [9]:
umrs_contexts = ['UMR']
umrs_bed_file = os.path.join(data_dir, "source", "UMRs.v5.bed.gz")
if not all([c in contexts_list for c in umrs_contexts]):
    from src.contexts import get_bed_contexts
    site_contexts_df = get_bed_contexts(site_contexts_df, umrs_bed_file, context_name='UMR')
    site_contexts_df.to_hdf(site_contexts_df_h5, key='df')

#### riboSNitch

In [10]:
ribosnitch_contexts = ['riboSNitch']
snpfold_results_file = os.path.join(data_dir, "source", "snpfold.results.csv.gz")
if not all([c in contexts_list for c in ribosnitch_contexts]):
    from src.contexts import get_ID_list_contexts
    snpfold_results_df = pd.read_csv(snpfold_results_file, sep=",", header=0)
    snpfold_results_df = snpfold_results_df[['variant_id', 'chrom', 'pos', 'CC_BPPROB']]
    snpfold_results_df.dropna(inplace=True, ignore_index=True)
    snpfold_results_df.rename(columns={'chrom': 'CHROM', 'pos': 'POS', 'variant_id': 'ID'}, inplace=True)
    # remove duplicates based on 'ID', keeping the smallest 'CC_BPPROB' value for each duplicate
    snpfold_results_df = snpfold_results_df.sort_values(by=['CC_BPPROB'], ascending=True).drop_duplicates(subset=['ID'], keep='first')
    snpfold_results_df.sort_values(by=['CHROM', 'POS'], inplace=True)
    snpfold_results_df['riboSNitch'] = snpfold_results_df.apply(lambda x: "riboSNitch" if x['CC_BPPROB'] < 0.8 else "non-riboSNitch", axis=1)
    riboSNitch_ID_list = snpfold_results_df[snpfold_results_df['riboSNitch'] == 'riboSNitch']['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, riboSNitch_ID_list, context_name='riboSNitch')
    site_contexts_df.to_hdf(site_contexts_df_h5, key='df')

#### Phylogenetic conservation

In [11]:
phyloP_file = os.path.join(data_dir, "source", "Sbi_PhyloP.csv.gz")
phylo_conservation_contexts = ['non_CEs', 'conserved_sites', 'neutral_sites', 'accelerated_sites']
if not all([c in contexts_list for c in phylo_conservation_contexts]):
    from src.contexts import get_ID_list_contexts
    phyloP_df = pd.read_csv(phyloP_file, sep=",", header=0)
    genic_sites_ID_list = site_contexts_df[site_contexts_df['genic_15kb'] == True]['ID'].tolist()
    # conserved_sites
    conserved_sites_ID_list = phyloP_df[phyloP_df['phyloP'] > 1]['ID'].tolist()
    conserved_sites_ID_list = set(conserved_sites_ID_list).intersection(set(genic_sites_ID_list))
    site_contexts_df = get_ID_list_contexts(site_contexts_df, conserved_sites_ID_list, context_name='conserved_sites')
    # neutral_sites
    neutral_sites_ID_list = phyloP_df[(phyloP_df['phyloP'] >= -1) & (phyloP_df['phyloP'] <= 1)]['ID'].tolist()
    neutral_sites_ID_list = set(neutral_sites_ID_list).intersection(set(genic_sites_ID_list))
    site_contexts_df = get_ID_list_contexts(site_contexts_df, neutral_sites_ID_list, context_name='neutral_sites')
    # accelerated_sites
    accelerated_sites_ID_list = phyloP_df[phyloP_df['phyloP'] < -1]['ID'].tolist()
    accelerated_sites_ID_list = set(accelerated_sites_ID_list).intersection(set(genic_sites_ID_list))
    site_contexts_df = get_ID_list_contexts(site_contexts_df, accelerated_sites_ID_list, context_name='accelerated_sites')
    # other sites
    non_CEs_ID_list = set(genic_sites_ID_list) - set(conserved_sites_ID_list) - set(neutral_sites_ID_list) - set(accelerated_sites_ID_list)
    site_contexts_df = get_ID_list_contexts(site_contexts_df, non_CEs_ID_list, context_name='non_CEs')

    site_contexts_df.to_hdf(site_contexts_df_h5, key='df')

#### Nucleotide Diversity (in Gene-neighborhood regions)

In [12]:
ref_gff_file = os.path.join(data_dir, "source", "Sbicolor_730_v5.1.gene_exons.gff3.gz")
ref_genome_fa = os.path.join(data_dir, "source", "Sbicolor_730_v5.0.fa.gz")
vcf_file = os.path.join(data_dir, "source", "landraces.vcf.gz")
pi_contexts = ['low_pi', 'med_pi', 'high_pi']
window_size = 500
gene_flank_size = 15000
min_snp_count = 3

if not all([c in contexts_list for c in pi_contexts]):
    from src.contexts import get_pi_statistics, get_window_contexts
    pi_df = get_pi_statistics(ref_gff_file, ref_genome_fa, vcf_file, window_size=window_size, gene_flank_size=gene_flank_size, min_snp_count=min_snp_count)
    low_threshold = pi_df['PI'].quantile(0.25)
    high_threshold = pi_df['PI'].quantile(0.75)
    low_pi_df = pi_df[pi_df['PI'] < low_threshold]
    med_pi_df = pi_df[(pi_df['PI'] >= low_threshold) & (pi_df['PI'] < high_threshold)]
    high_pi_df = pi_df[pi_df['PI'] >= high_threshold]
    site_contexts_df = get_window_contexts(site_contexts_df, low_pi_df, 'low_pi')
    site_contexts_df = get_window_contexts(site_contexts_df, med_pi_df, 'med_pi')
    site_contexts_df = get_window_contexts(site_contexts_df, high_pi_df, 'high_pi')
    site_contexts_df.to_hdf(site_contexts_df_h5, key='df')

#### Minor Allele Frequency (MAF)

In [13]:
maf_contexts = ['maf_0.05_0.1', 'maf_0.1_0.2', 'maf_0.2_0.3', 'maf_0.3_0.4', 'maf_0.4_0.5']

if not all([c in contexts_list for c in maf_contexts]):
    from src.contexts import get_ID_list_contexts
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.05) & (site_contexts_df['MAF'] < 0.1)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.05_0.1')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.1) & (site_contexts_df['MAF'] < 0.2)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.1_0.2')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.2) & (site_contexts_df['MAF'] < 0.3)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.2_0.3')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.3) & (site_contexts_df['MAF'] < 0.4)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.3_0.4')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.4) & (site_contexts_df['MAF'] <= 0.5)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.4_0.5')
    site_contexts_df.to_hdf(site_contexts_df_h5, key='df')

### 1.3 对CDS中 MAF大于0.01的位点进行context注释

获取位点列表

In [14]:
from src.contexts import load_site_info_df
import pandas as pd

site_contexts_df_h5 = os.path.join(data_dir, "results", f"site_contexts.cds.maf0.01.h5")

if not have_file(site_contexts_df_h5):
    site_contexts_df = load_site_info_df(cds_output_vcf_file_stat_h5)
    site_contexts_df = site_contexts_df[['ID', 'CHROM', 'POS', 'MAF']]
    site_contexts_df.to_hdf(site_contexts_df_h5, key='df')
else:
    site_contexts_df = pd.read_hdf(site_contexts_df_h5, key='df')

contexts_list = [i for i in site_contexts_df.columns.tolist() if i not in ['ID', 'CHROM', 'POS', 'MAF']]
print(f" {len(contexts_list)} contexts have already been recorded in site_contexts_df")

site_contexts_df

 10 contexts have already been recorded in site_contexts_df


,ID,CHROM,POS,MAF,missense,synonymous,putative_LoF,maf_0.01_0.03_cds_0.01,maf_0.03_0.05_cds_0.01,maf_0.05_0.1_cds_0.01,maf_0.1_0.2_cds_0.01,maf_0.2_0.3_cds_0.01,maf_0.3_0.4_cds_0.01,maf_0.4_0.5_cds_0.01
0,SNP_Chr01_22482,Chr01,22482,0.019187,True,False,False,True,False,False,False,False,False,False
1,SNP_Chr01_22585,Chr01,22585,0.019187,False,True,False,True,False,False,False,False,False,False
2,SNP_Chr01_22703,Chr01,22703,0.019187,True,False,False,True,False,False,False,False,False,False
3,SNP_Chr01_33180,Chr01,33180,0.024831,False,True,False,True,False,False,False,False,False,False
4,SNP_Chr01_34142,Chr01,34142,0.382619,True,False,False,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
409052,SNP_Chr10_62837224,Chr10,62837224,0.051919,False,True,False,False,False,True,False,False,False,False
409053,SNP_Chr10_62841974,Chr10,62841974,0.453725,True,False,False,False,False,False,False,False,False,True
409054,SNP_Chr10_62842354,Chr10,62842354,0.016930,False,True,False,True,False,False,False,False,False,False
409055,INDEL_Chr10_62842730,Chr10,62842730,0.030474,False,False,True,False,True,False,False,False,False,False


#### Coding Consequences

In [15]:
coding_consequence_contexts = ['missense_cds_0.01', 'synonymous_cds_0.01', 'putative_LoF_cds_0.01']
cds_snpeff_vcf_file = cds_output_vcf_file
if not all([c in contexts_list for c in coding_consequence_contexts]):
    from src.contexts import get_coding_mutation_contexts
    site_contexts_df = get_coding_mutation_contexts(site_contexts_df, cds_snpeff_vcf_file, dataset='_cds_0.01')
    site_contexts_df.to_hdf(site_contexts_df_h5, key='df')

#### Minor Allele Frequency (MAF)

In [16]:
maf_contexts = ['maf_0.01_0.03_cds_0.01', 'maf_0.03_0.05_cds_0.01', 'maf_0.05_0.1_cds_0.01', 'maf_0.1_0.2_cds_0.01', 'maf_0.2_0.3_cds_0.01', 'maf_0.3_0.4_cds_0.01', 'maf_0.4_0.5_cds_0.01']

if not all([c in contexts_list for c in maf_contexts]):
    from src.contexts import get_ID_list_contexts
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.01) & (site_contexts_df['MAF'] < 0.03)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.01_0.03_cds_0.01')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.03) & (site_contexts_df['MAF'] < 0.05)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.03_0.05_cds_0.01')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.05) & (site_contexts_df['MAF'] < 0.1)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.05_0.1_cds_0.01')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.1) & (site_contexts_df['MAF'] < 0.2)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.1_0.2_cds_0.01')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.2) & (site_contexts_df['MAF'] < 0.3)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.2_0.3_cds_0.01')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.3) & (site_contexts_df['MAF'] < 0.4)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.3_0.4_cds_0.01')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.4) & (site_contexts_df['MAF'] <= 0.5)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.4_0.5_cds_0.01')
    site_contexts_df.to_hdf(site_contexts_df_h5, key='df')

## 2. 定义gene-level contexts

获得基因列表

In [17]:
import pandas as pd
from src.utils import have_file

ref_gff_file = os.path.join(data_dir, "source", "Sbicolor_730_v5.1.gene_exons.gff3.gz")
gene_contexts_df_h5 = os.path.join(data_dir, "results", f"gene_contexts.h5")

if not have_file(gene_contexts_df_h5):
    from src.contexts import load_gene_info_df
    gene_contexts_df = load_gene_info_df(ref_gff_file)
    gene_contexts_df.to_hdf(gene_contexts_df_h5, key='df')
else:
    gene_contexts_df = pd.read_hdf(gene_contexts_df_h5, key='df')

contexts_list = [i for i in gene_contexts_df.columns.tolist() if i not in ['ID', 'CHROM', 'START', 'END', 'STRAND']]
print(f" {len(contexts_list)} contexts have already been recorded in gene_contexts_df")

gene_contexts_df

 13 contexts have already been recorded in gene_contexts_df


,ID,CHROM,START,END,STRAND,flowering_time_genes,no_expression_genes,high_variance_genes,stably_expressed_genes,module_hub_genes,module_genes,non_module_genes,orphan_genes,single_copy_genes,multi_copy_genes,low_recombination_genes,med_recombination_genes,high_recombination_genes
0,Sobic.001G000100.v5.1,Chr01,22085,23338,+,False,False,False,False,False,True,False,False,True,False,True,False,False
1,Sobic.001G000200.v5.1,Chr01,31590,35375,-,False,False,False,False,True,False,False,False,True,False,True,False,False
2,Sobic.001G000300.v5.1,Chr01,43890,44492,-,False,False,False,False,False,True,False,False,False,True,True,False,False
3,Sobic.001G000400.v5.1,Chr01,44828,63076,-,False,False,False,False,False,True,False,False,True,False,True,False,False
4,Sobic.001G000700.v5.1,Chr01,80708,83931,+,False,False,False,False,False,True,False,False,False,True,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31814,Sobic.010G279800.v5.1,Chr10,62816416,62825451,+,False,False,False,False,False,True,False,False,True,False,False,False,False
31815,Sobic.010G279900.v5.1,Chr10,62825908,62834521,-,False,False,False,False,False,True,False,False,True,False,False,False,False
31816,Sobic.010G280000.v5.1,Chr10,62836480,62837370,-,False,False,False,False,False,True,False,False,True,False,False,False,False
31817,Sobic.010G280100.v5.1,Chr10,62838030,62841326,+,False,False,False,False,False,True,False,False,True,False,False,False,False


#### Flowering time genes

In [18]:
flowering_gene_contexts = ['flowering_time_genes']

if not all([c in contexts_list for c in flowering_gene_contexts]):
    from src.contexts import get_ID_list_contexts
    flowering_genes_file = os.path.join(data_dir, "source", "Sorghum_flowering_genes.csv")
    flowering_genes_df = pd.read_csv(flowering_genes_file, sep=",", header=0)
    flowering_gene_IDs = flowering_genes_df['Gene_ID'].to_list()
    gene_contexts_df = get_ID_list_contexts(gene_contexts_df, flowering_gene_IDs, context_name='flowering_time_genes')
    gene_contexts_df.to_hdf(gene_contexts_df_h5, key='df')

#### Gene Expression

In [19]:
expression_gene_contexts = ['no_expression_genes', 'high_variance_genes', 'stably_expressed_genes']

if not all([c in contexts_list for c in expression_gene_contexts]):
    from src.contexts import get_expression_contexts
    TPMs_csv_file = os.path.join(data_dir, "source", "Sbi_NCBI_SRA_TPMs.csv.gz")
    gene_contexts_df = get_expression_contexts(gene_contexts_df, TPMs_csv_file)
    gene_contexts_df.to_hdf(gene_contexts_df_h5, key='df')

#### Co-expression Networks

In [20]:
GCNs_gene_contexts = ['non_module_genes', 'module_genes', 'module_hub_genes']

if not all([c in contexts_list for c in GCNs_gene_contexts]):
    from src.contexts import get_GCNs_contexts
    module_file = os.path.join(data_dir, "source", "edge_weight_N3.0.01.clusterone.txt.gz")
    edge_weight_file = os.path.join(data_dir, "source", "edge_weight_N3.0.01.tsv.gz")
    gene_contexts_df = get_GCNs_contexts(gene_contexts_df, edge_weight_file, module_file)
    gene_contexts_df.to_hdf(gene_contexts_df_h5, key='df')

#### Gene families

In [21]:
gene_family_contexts = ['orphan_genes', 'single_copy_genes', 'multi_copy_genes']

if not all([c in contexts_list for c in gene_family_contexts]):
    from src.contexts import get_gene_family_contexts
    gene_family_file = os.path.join(data_dir, "source", "Poaceae_HOG.tsv.gz")
    gene_contexts_df = get_gene_family_contexts(gene_contexts_df, gene_family_file)
    gene_contexts_df.to_hdf(gene_contexts_df_h5, key='df')

#### Recombination context

In [22]:
recombination_contexts = ['low_recombination_genes', 'med_recombination_genes', 'high_recombination_genes']

if not all([c in contexts_list for c in recombination_contexts]):
    from src.contexts import get_recombination_contexts
    recombination_file = os.path.join(data_dir, "source", "avg_cM_genomewide_NAM_population.csv.gz")
    gene_contexts_df = get_recombination_contexts(gene_contexts_df, recombination_file)
    gene_contexts_df.to_hdf(gene_contexts_df_h5, key='df')

## 3. 统计各种context的数量

In [2]:
import json

datasets = {
    'genome': os.path.join(data_dir, "results", "site_contexts.genome.maf0.05.h5"), # All variants with MAF >= 0.05
    'cds': os.path.join(data_dir, "results", "site_contexts.cds.maf0.01.h5"), # All variants in coding regions with MAF >= 0.01
    'gene': os.path.join(data_dir, "results", "gene_contexts.h5") # All genes
}

contexts_settings_json_file = os.path.join(os.getcwd(), "config", "contexts_settings.json")
with open(contexts_settings_json_file, "r") as f:
    contexts_settings = json.load(f)

### 3.1 全基因组位点中MAF大于0.05的位点contexts统计

In [ ]:
import numpy as np
import pandas as pd

dataset = 'genome'
context_df = pd.read_hdf(datasets[dataset], key='df')
contexts_list = contexts_settings[dataset]

print(f"{len(contexts_list)} contexts have been listed in contexts_settings.json for dataset '{dataset}'")
print( f" dataset '{dataset}': {context_df.shape[0]:,} Sites/Genes")
for context in contexts_list:
    print(f" {context}: {np.sum(context_df[context]).astype(int):,} Sites/Genes")

30 contexts have been listed in contexts_settings.json for dataset 'genome'
 dataset 'genome': 6,452,578 Sites/Genes
 gene_body: 1,013,897 Sites/Genes
 genic_2kb: 2,277,703 Sites/Genes
 genic_proximal: 3,077,170 Sites/Genes
 genic_15kb: 3,938,903 Sites/Genes
 TE: 3,736,514 Sites/Genes
 other_intergenic: 1,054,970 Sites/Genes
 upstream_2kb: 888,419 Sites/Genes
 five_prime_UTR: 87,769 Sites/Genes
 CDS: 240,922 Sites/Genes
 intron: 520,464 Sites/Genes
 three_prime_UTR: 155,961 Sites/Genes
 downstream_2kb: 755,968 Sites/Genes
 missense: 112,419 Sites/Genes
 synonymous: 113,885 Sites/Genes
 putative_LoF: 6,966 Sites/Genes
 ACR: 104,520 Sites/Genes
 UMR: 1,313,439 Sites/Genes
 riboSNitch: 240,777 Sites/Genes
 low_pi: 162,987 Sites/Genes
 med_pi: 1,510,914 Sites/Genes
 high_pi: 2,252,023 Sites/Genes
 non_CEs: 2,043,225 Sites/Genes
 conserved_sites: 80,583 Sites/Genes
 neutral_sites: 1,630,480 Sites/Genes
 accelerated_sites: 184,615 Sites/Genes
 maf_0.05_0.1: 1,827,153 Sites/Genes
 maf_0.1_0.2

### 3.2 CDS中MAF大于0.01的位点contexts统计

In [4]:
import numpy as np
import pandas as pd

dataset = 'cds'
context_df = pd.read_hdf(datasets[dataset], key='df')
contexts_list = contexts_settings[dataset]

print(f"{len(contexts_list)} contexts have been listed in contexts_settings.json for dataset '{dataset}'")
print( f" dataset '{dataset}': {context_df.shape[0]:,} Sites/Genes")
for context in contexts_list:
    print(f" {context}: {np.sum(context_df[context]).astype(int):,} Sites/Genes")

10 contexts have been listed in contexts_settings.json for dataset 'cds'
 dataset 'cds': 409,057 Sites/Genes
 missense_cds_0.01: 197,917 Sites/Genes
 synonymous_cds_0.01: 182,584 Sites/Genes
 putative_LoF_cds_0.01: 14,191 Sites/Genes
 maf_0.01_0.03_cds_0.01: 109,506 Sites/Genes
 maf_0.03_0.05_cds_0.01: 48,525 Sites/Genes
 maf_0.05_0.1_cds_0.01: 60,275 Sites/Genes
 maf_0.1_0.2_cds_0.01: 73,086 Sites/Genes
 maf_0.2_0.3_cds_0.01: 45,983 Sites/Genes
 maf_0.3_0.4_cds_0.01: 36,858 Sites/Genes
 maf_0.4_0.5_cds_0.01: 34,824 Sites/Genes


### 3.3 Gene-level contexts统计

In [5]:
import numpy as np
import pandas as pd

dataset = 'gene'
context_df = pd.read_hdf(datasets[dataset], key='df')
contexts_list = contexts_settings[dataset]

print(f"{len(contexts_list)} contexts have been listed in contexts_settings.json for dataset '{dataset}'")
print( f" dataset '{dataset}': {context_df.shape[0]:,} Sites/Genes")
for context in contexts_list:
    print(f" {context}: {np.sum(context_df[context]).astype(int):,} Sites/Genes")

13 contexts have been listed in contexts_settings.json for dataset 'gene'
 dataset 'gene': 31,819 Sites/Genes
 flowering_time_genes: 312 Sites/Genes
 no_expression_genes: 1,232 Sites/Genes
 high_variance_genes: 5,195 Sites/Genes
 stably_expressed_genes: 1,350 Sites/Genes
 non_module_genes: 4,267 Sites/Genes
 module_genes: 20,422 Sites/Genes
 module_hub_genes: 7,130 Sites/Genes
 orphan_genes: 3,237 Sites/Genes
 multi_copy_genes: 12,421 Sites/Genes
 single_copy_genes: 16,161 Sites/Genes
 high_recombination_genes: 7,919 Sites/Genes
 low_recombination_genes: 7,919 Sites/Genes
 med_recombination_genes: 15,836 Sites/Genes
